In [ ]:
import requests
import pandas as pd
import time

import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv('TOUR_API_KEY')  # 공공데이터포털 한국관광공사 TourAPI 키 (.env)
BASE_URL = 'http://apis.data.go.kr/B551011/KorService1/areaBasedList1'

def fetch_jeju_attractions(area_code=39, content_type_id=12, num_of_rows=100):
    all_items = []
    page_no = 1

    while True:
        params = {
            'serviceKey': API_KEY,
            'numOfRows': num_of_rows,
            'pageNo': page_no,
            'MobileOS': 'ETC',
            'MobileApp': 'JejuTourApp',
            '_type': 'json',
            'areaCode': area_code,
            'contentTypeId': content_type_id,
            'arrange': 'A',
        }

        resp = requests.get(BASE_URL, params=params)
        resp.raise_for_status()
        data = resp.json()

        body = data.get('response', {}).get('body', {})
        total_count = body.get('totalCount', 0)
        items = body.get('items', {})

        if not items:
            break

        item_list = items.get('item', [])
        if isinstance(item_list, dict):
            item_list = [item_list]

        all_items.extend(item_list)
        print(f'페이지 {page_no} | 수집: {len(all_items)} / 전체: {total_count}')

        if len(all_items) >= total_count:
            break

        page_no += 1
        time.sleep(0.3)

    return all_items

items = fetch_jeju_attractions()
df = pd.DataFrame(items)
print(f'총 {len(df)}개 수집 완료')
df.head()

In [ ]:
cols = ['contentid', 'contenttypeid', 'title', 'addr1', 'addr2',
        'mapx', 'mapy', 'areacode', 'sigungucode', 'firstimage', 'tel']
cols_exist = [c for c in cols if c in df.columns]

df_save = df[cols_exist].copy()
df_save.to_csv('jeju_attractions_api.csv', index=False, encoding='utf-8-sig')
print('저장 완료: jeju_attractions_api.csv')
df_save